# Ekstraksi pesan dan rute RPL dari CSV Cooja
1. Jalankan sel di bawah, lalu unggah **satu CSV atau CSV.GZ** dengan kolom `time,node,message`.
2. `SNAPSHOT_SECONDS=None` memakai parent terakhir; ganti menjadi `600.0` untuk detik ke-600. Timeline pesan tetap mencakup seluruh log.
3. Hasil CSV diunduh dalam ZIP. Untuk memakai file di Drive yang sudah di-mount, isi `INPUT_PATH` dengan path lengkap.

**Batas interpretasi:** rantai parent bukan trace paket aktual. Kejadian DAO bukan dump tabel downward aktif. Kandidat pasangan DIO hanya heuristik waktu. UDP tanpa pasangan bukan otomatis paket hilang. Pemetaan global IPv6 memakai kesamaan IID 64-bit yang unik dan dicatat sebagai inferensi.

Kode diuji lokal pada DTWRL101.csv; antarmuka upload/download Colab dijalankan saat notebook dibuka di Colab. Tidak memerlukan paket tambahan untuk parsing dan ekspor CSV.


In [12]:
"""Ekstraksi log RPL untuk Google Colab (CSV time,node,message).

Salin seluruh kode ke satu sel Colab lalu jalankan, atau upload file ini
dan jalankan: %run ekstrak_routing_colab.py
Hanya upload satu run CSV per eksekusi. Jangan gabungkan beberapa run.
Output: snapshot parent, timeline kontrol, UDP, kandidat pasangan DIO,
pasangan UDP, kejadian route DAO, diagnostik, dan ZIP.
Tidak mengklaim tabel downward aktif atau jalur aktual per-hop paket.
"""
import csv
import gzip
import ipaddress
import re
import json
import zipfile
from pathlib import Path
from datetime import datetime
from collections import defaultdict, Counter
from bisect import bisect_left, bisect_right

# UBAH BAGIAN INI
INPUT_PATH =    '/content/drive/MyDrive/LOG CONTIKI-NG/DTW-RL/3 Juli:Trajektori Berubah/CSV/dtwrlcomplete/Mob5/30/DTWRL301.csv'       # None: pilih file melalui upload Google Colab
SNAPSHOT_SECONDS = 3600    # None: akhir log; contoh 600.0 = detik ke-600
ROOT_NODE = None           # None: otomatis dari pesan DAG root
DIO_MATCH_WINDOW_S = 1.0   # Jendela pencarian kandidat TX sebelum RX DIO (heuristik)
UDP_MATCH_WINDOW_S = 120.0 # Batas pencarian TX sebelum RX untuk source + sequence sama
WRAP_SECONDS = 3600        # Format MM:SS.mmm yang kembali ke 00:00 setelah 1 jam
                          # Gunakan None bila timestamp tidak mengalami rollover.


def ip(value):
    return str(ipaddress.IPv6Address(value.strip().rstrip(',;')))


def read_log(path, wrap_seconds=3600):
    opener = gzip.open if str(path).endswith('.gz') else open
    rows = []
    offset, previous = 0.0, None
    with opener(path, 'rt', encoding='utf-8-sig', newline='') as handle:
        reader = csv.DictReader(handle)
        if not {'time', 'node', 'message'} <= set(reader.fieldnames or []):
            raise ValueError('CSV harus memiliki kolom time,node,message.')
        for record, raw in enumerate(reader, 1):
            if None in raw or any(raw[k] is None for k in ('time', 'node', 'message')):
                raise ValueError(f'Rekaman {record}: CSV tidak valid; periksa quoting koma.')
            parts = raw['time'].strip().split(':')
            if len(parts) == 2:
                base = 60 * int(parts[0]) + float(parts[1])
            elif len(parts) == 3:
                base = 3600 * int(parts[0]) + 60 * int(parts[1]) + float(parts[2])
            elif len(parts) == 1:
                base = float(parts[0])  # angka tanpa titik dua dianggap detik
            else:
                raise ValueError(f'Format waktu tidak dikenal: {raw["time"]}')
            elapsed = base + offset
            if previous is not None and elapsed < previous - 1e-6:
                # Hanya izinkan rollover dekat batas jam, bukan reset/merge run.
                old_base = previous - offset
                if (len(parts) == 2 and wrap_seconds and
                        old_base >= wrap_seconds - 60 and base <= 60):
                    offset += wrap_seconds
                    elapsed = base + offset
                else:
                    raise ValueError(f'Waktu mundur pada rekaman {record}. '
                                     'Periksa urutan CSV atau run yang tercampur.')
            rows.append(dict(record=record, time_raw=raw['time'], time_s=elapsed,
                             node=int(raw['node']), message=raw['message']))
            previous = elapsed
    if not rows:
        raise ValueError('CSV kosong.')
    return rows


def analyse(rows, snapshot=None, root_override=None):
    end = rows[-1]['time_s']
    cut = end if snapshot is None else float(snapshot)
    if not 0 <= cut <= end:
        raise ValueError(f'Snapshot harus antara 0 dan {end:.3f} detik.')
    selected = [r for r in rows if r['time_s'] <= cut]
    addresses, roots, parent_events, route_events, diagnostics = {}, set(), [], [], []
    nodes = {r['node'] for r in selected}
    for r in selected:
        msg, node = r['message'], r['node']
        for pattern in (r'Tentative link-local IPv6 address:\s*(\S+)',
                        r'Node set to be a DAG root with DAG ID\s*(\S+)'):
            m = re.search(pattern, msg)
            if m:
                address = ip(m[1])
                if address in addresses and addresses[address] != node:
                    raise ValueError(f'Alamat {address} digunakan beberapa node.')
                addresses[address] = node
        if 'Node set to be a DAG root' in msg:
            roots.add(node)
        m = re.search(r'rpl_set_preferred_parent (\S+) used to be (\S+)', msg)
        if m:
            parent_events.append({**r, 'parent_ipv6': '' if m[1] == 'NULL' else ip(m[1]),
                                  'old_parent_ipv6': '' if m[2] == 'NULL' else ip(m[2])})
        m = re.search(r'Added a route to (\S+) via (\S+)', msg)
        if m:
            route_events.append({**r, 'event': 'route_added', 'destination': m[1],
                                 'next_hop_ipv6': ip(m[2])})
        elif 'No more routes to ' in msg:
            route_events.append({**r, 'event': 'no_path_notice' if 'No-Path DAO' in msg
                                 else 'no_more_routes',
                                 'destination': msg.split('No more routes to ')[1].split()[0],
                                 'next_hop_ipv6': ''})
        elif re.search(r'route|routing', msg, re.I):
            route_events.append({**r, 'event': 'other_route_message',
                                 'destination': '', 'next_hop_ipv6': ''})
        if ('No space for more route entries' in msg or
                'Could not add a route' in msg or 'Starting Contiki' in msg):
            diagnostics.append(r)
    if root_override is None:
        if len(roots) != 1:
            raise ValueError(f'Root terdeteksi: {sorted(roots)}. Isi ROOT_NODE secara eksplisit.')
        root = next(iter(roots))
    else:
        root = int(root_override)
    # Restart di tengah run membuat carry-forward parent tidak valid.
    starts = {}
    for r in diagnostics:
        if 'Starting Contiki' in r['message']:
            starts[r['node']] = starts.get(r['node'], 0) + 1
    if any(n > 1 for n in starts.values()):
        raise ValueError('Terdeteksi restart node. Pisahkan episode sebelum rekonstruksi.')
    latest = {}
    for e in parent_events:
        e['parent_node'] = addresses.get(e['parent_ipv6'], '')
        e['old_parent_node'] = addresses.get(e['old_parent_ipv6'], '')
        latest[e['node']] = e
    def trace(node):
        chain, seen = [node], {node}
        while chain[-1] != root:
            cur = chain[-1]
            if cur not in latest:
                return chain, 'parent_belum_tercatat'
            event = latest[cur]
            if not event['parent_ipv6']:
                return chain, 'parent_NULL'
            p = event['parent_node']
            if p == '':
                return chain, 'alamat_parent_belum_terpetakan'
            chain.append(p)
            if p in seen:
                return chain, 'siklus_parent'
            seen.add(p)
        return chain, 'root' if node == root else 'rantai_menuju_root'
    table = []
    for node in sorted(nodes | {root}):
        chain, status = trace(node)
        e = latest.get(node, {})
        table.append(dict(snapshot_s=cut, node=node, preferred_parent=e.get('parent_node', ''),
                          parent_ipv6=e.get('parent_ipv6', ''),
                          path_to_root=' -> '.join(map(str, chain)),
                          hops=len(chain)-1 if status in ('root','rantai_menuju_root') else '',
                          status=status, last_parent_event_s=e.get('time_s', ''),
                          evidence_record=e.get('record', '')))
    summary = dict(root=root, snapshot_s=cut, log_end_s=end,
                   nodes_observed=len(nodes), parent_events=len(parent_events),
                   chains_to_root=sum(x['status']=='rantai_menuju_root' for x in table),
                   route_capacity_errors=sum('No space for more route entries' in r['message']
                                             for r in diagnostics))
    return table, parent_events, route_events, diagnostics, addresses, summary


def write_csv(path, rows, fields):
    with open(path, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def extract_messages(rows, addresses, parents, root, dio_window=1.0, udp_window=120.0):
    """Semua pencocokan mempertahankan nomor rekaman sebagai bukti.

    TX/RX adalah catatan logger, bukan bukti tangkapan radio. Pesan MMDEW-CTRL
    disimpan sebagai marker terpisah agar tidak menggandakan hitungan TX.
    """
    if dio_window <= 0 or udp_window <= 0:
        raise ValueError('Jendela pencocokan harus positif.')
    addresses = dict(addresses)
    # Global IPv6 dipetakan lewat kesamaan 64-bit interface identifier (IID)
    # dengan link-local startup, bukan digit terakhir atau node pengirim DAO.
    # DAO/No-Path dapat mengiklankan tujuan lain, bukan alamat pengirim.
    iid_nodes = defaultdict(set)
    for address, node in addresses.items():
        iid_nodes[int(ipaddress.IPv6Address(address)) & ((1 << 64)-1)].add(node)
    for r in rows:
        for address in re.findall(r'(?:prefix |From )([0-9a-fA-F:]+)', r['message']):
            address = ip(address)
            candidates = iid_nodes[int(ipaddress.IPv6Address(address)) & ((1 << 64)-1)]
            if address not in addresses and len(candidates) == 1:
                addresses[address] = next(iter(candidates))
    controls, udp, markers, diagnostics = [], [], [], []
    state, parent_index = {}, 0

    def chain_at_send(node):
        chain, seen = [node], {node}
        while chain[-1] != root:
            e = state.get(chain[-1])
            if e is None:
                return ' -> '.join(map(str, chain)), 'parent_belum_tercatat'
            if not e['parent_ipv6']:
                return ' -> '.join(map(str, chain)), 'parent_NULL'
            p = addresses.get(e['parent_ipv6'])
            if p is None:
                return ' -> '.join(map(str, chain)), 'alamat_parent_belum_terpetakan'
            chain.append(p)
            if p in seen:
                return ' -> '.join(map(str, chain)), 'siklus_parent'
            seen.add(p)
        return ' -> '.join(map(str, chain)), 'rekonstruksi_parent_bukan_trace_paket'

    for r in rows:
        msg, node = r['message'], r['node']
        while parent_index < len(parents) and parents[parent_index]['record'] <= r['record']:
            e = parents[parent_index]
            state[e['node']] = e
            parent_index += 1
        if 'MMDEW-CTRL:' in msg:
            markers.append(r)
            continue
        if re.search(r'\[ERR\s*:|\[WARN\s*:|Starting Contiki', msg):
            diagnostics.append(r)
        protocol = ('DAO-ACK' if re.search(r'DAO[- ]ACK', msg, re.I) else
                    'DAO' if re.search(r'\bDAO\b', msg) else
                    'DIO' if re.search(r'\bDIO\b', msg) else
                    'DIS' if re.search(r'\bDIS\b', msg) else '')
        if protocol:
            e = {**r, 'protocol': protocol, 'event': 'OTHER', 'source_node': '',
                 'destination_node': '', 'source_ipv6': '', 'destination_ipv6': '',
                 'mode': '', 'rank': '', 'sequence': '', 'prefix': '', 'lifetime': ''}
            # Pemilihan pola ketat menghindari menghitung "sending DAO" timer
            # atau "Received consistent DIO" sebagai paket tambahan.
            m = re.search(r'Sending a multicast-DIO with rank (\d+)', msg)
            u = re.search(r'Sending unicast-DIO with rank (\d+) to (\S+)', msg)
            rx = re.search(r'Received a (DIO|DIS|DAO) from (\S+)', msg)
            dis = re.search(r'Sending a DIS to (\S+)', msg)
            dao = re.search(r'Sending a (No-Path )?DAO with sequence number (\d+), '
                            r'lifetime (\d+), prefix (\S+) to (\S+)', msg)
            if m:
                e.update(event='TX', source_node=node, mode='multicast', rank=int(m[1]))
            elif u:
                e.update(event='TX', source_node=node, mode='unicast', rank=int(u[1]),
                         destination_ipv6=ip(u[2]))
            elif rx:
                e.update(event='RX', destination_node=node, source_ipv6=ip(rx[2]))
            elif dis:
                addr = ip(dis[1])
                e.update(event='TX', source_node=node, destination_ipv6=addr,
                         mode='multicast' if ipaddress.IPv6Address(addr).is_multicast else 'unicast')
            elif dao:
                e.update(event='TX_NO_PATH' if dao[1] else 'TX', source_node=node,
                         destination_ipv6=ip(dao[5]), sequence=int(dao[2]),
                         lifetime=int(dao[3]), prefix=dao[4], mode='unicast')
            elif protocol == 'DAO-ACK':
                # Format ACK belum terverifikasi pada CSV contoh. Simpan utuh.
                e['event'] = 'ACK_MESSAGE_UNPARSED'
            elif 'No-Path DAO received' in msg:
                e.update(event='NO_PATH_NOTICE', destination_node=node)
            if e['source_ipv6']:
                e['source_node'] = addresses.get(e['source_ipv6'], '')
            if e['destination_ipv6']:
                e['destination_node'] = addresses.get(e['destination_ipv6'], '')
            controls.append(e)
        sent = re.search(r'UDP-Client\].*?Sent Seq=(\d+)', msg)
        received = re.search(r'UDP-Server\].*?From (\S+), Seq=(\d+)', msg)
        if sent or received:
            e = {**r, 'event': 'TX' if sent else 'RX', 'source_node': node if sent else '',
                 'source_ipv6': '', 'destination_node': '' if sent else node,
                 'sequence': int(sent[1] if sent else received[2]), 'reported_delay_ms': '',
                 'parent_chain_at_tx': '', 'chain_status': ''}
            if received:
                e['source_ipv6'] = ip(received[1])
                e['source_node'] = addresses.get(e['source_ipv6'], '')
                delay = re.search(r'\bDelay=(\d+(?:\.\d+)?)ms', msg)
                if delay:
                    e['reported_delay_ms'] = float(delay[1])
            else:
                e['parent_chain_at_tx'], e['chain_status'] = chain_at_send(node)
            udp.append(e)
        elif 'UDP_SUMMARY' in msg or 'FINAL_STATS' in msg:
            udp.append({**r, 'event': 'SUMMARY'})
        elif 'udp: no matching connection found' in msg:
            udp.append({**r, 'event': 'UDP_NO_CONNECTION'})

    # DIO: tampilkan SEMUA kandidat dalam jendela, bukan pilih terdekat paksa.
    tx_by_node = defaultdict(list)
    for e in controls:
        if e['protocol'] == 'DIO' and e['event'] == 'TX':
            tx_by_node[e['source_node']].append(e)
    dio_times = {n: [e['time_s'] for e in es] for n, es in tx_by_node.items()}
    dio_matches = []
    for rx in controls:
        if rx['protocol'] != 'DIO' or rx['event'] != 'RX':
            continue
        txs = tx_by_node.get(rx['source_node'], [])
        times = dio_times.get(rx['source_node'], [])
        candidates = [tx for tx in txs[bisect_left(times, rx['time_s']-dio_window):
                                        bisect_right(times, rx['time_s'])]
                      if tx['record'] < rx['record'] and
                      (tx['mode'] == 'multicast' or tx['destination_node'] == rx['node'])]
        status = ('source_belum_terpetakan' if rx['source_node'] == '' else
                  'tidak_ada_kandidat' if not candidates else
                  'kandidat_tunggal_heuristik' if len(candidates) == 1 else 'ambigu')
        dio_matches.append(dict(rx_record=rx['record'], rx_time_s=rx['time_s'],
                                source_node=rx['source_node'], receiver_node=rx['node'],
                                candidate_count=len(candidates), status=status,
                                candidate_tx_records=';'.join(str(x['record']) for x in candidates),
                                candidate_tx_times_s=';'.join(str(x['time_s']) for x in candidates)))

    # UDP: cocokkan sumber + sequence, dengan batas waktu eksplisit.
    tx_udp = defaultdict(list)
    for e in udp:
        if e['event'] == 'TX':
            tx_udp[(e['source_node'], e['sequence'])].append(e)
    matches, matched_tx, rx_counts = [], set(), Counter()
    for rx in udp:
        if rx['event'] != 'RX':
            continue
        candidates = [tx for tx in tx_udp.get((rx['source_node'], rx['sequence']), [])
                      if tx['record'] < rx['record'] and
                      0 <= rx['time_s']-tx['time_s'] <= udp_window]
        e = dict(rx_record=rx['record'], source_node=rx['source_node'],
                 receiver_node=rx['node'], sequence=rx['sequence'], rx_time_s=rx['time_s'],
                 candidate_count=len(candidates), tx_record='', tx_time_s='',
                 log_time_difference_ms='', reported_delay_ms=rx['reported_delay_ms'],
                 direction='', status='source_belum_terpetakan' if rx['source_node']=='' else
                 'tidak_ada_kandidat' if not candidates else 'ambigu')
        if len(candidates) == 1:
            tx = candidates[0]
            matched_tx.add(tx['record']); rx_counts[tx['record']] += 1
            e.update(tx_record=tx['record'], tx_time_s=tx['time_s'],
                     log_time_difference_ms=round((rx['time_s']-tx['time_s'])*1000, 6),
                     status='cocok_source_sequence_waktu',
                     direction='upward_ke_root' if rx['node']==root and tx['node']!=root else
                     'downward_dari_root' if tx['node']==root and rx['node']!=root else
                     'lainnya')
        matches.append(e)
    for e in matches:
        e['rx_records_for_same_tx'] = rx_counts.get(e['tx_record'], '')
    unmatched = [{**e, 'status': 'tidak_ada_RX_unik_yang_cocok_bukan_bukti_packet_loss'}
                 for es in tx_udp.values() for e in es if e['record'] not in matched_tx]
    stats = [{'protocol': p, 'event': ev, 'count': n}
             for (p, ev), n in sorted(Counter((e['protocol'],e['event']) for e in controls).items())]
    return dict(controls=controls, udp=udp, markers=markers, diagnostics=diagnostics,
                dio_matches=dio_matches, udp_matches=matches, udp_unmatched=unmatched,
                control_counts=stats, addresses=addresses)


def main():
    source = INPUT_PATH
    if source is None:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Upload tepat satu CSV atau CSV.GZ untuk satu run.')
        source = next(iter(uploaded))
    rows = read_log(source, WRAP_SECONDS)
    table, parents, routes, diagnostics, addresses, summary = analyse(
        rows, SNAPSHOT_SECONDS, ROOT_NODE)
    # Snapshot memengaruhi tabel parent saja; timeline mencakup seluruh log.
    _, all_parents, all_routes, _, all_addresses, full_summary = analyse(rows, None, ROOT_NODE)
    extracted = extract_messages(rows, all_addresses, all_parents, full_summary['root'],
                                 DIO_MATCH_WINDOW_S, UDP_MATCH_WINDOW_S)
    out = Path('hasil_routing') / (Path(source).name.replace('.', '_') + '_' +
                                  datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
    out.mkdir(parents=True, exist_ok=False)
    common = ['record', 'time_raw', 'time_s', 'node', 'message']
    write_csv(out/'parent_snapshot.csv', table, list(table[0]))
    write_csv(out/'parent_events.csv', all_parents, common +
              ['parent_ipv6','parent_node','old_parent_ipv6','old_parent_node'])
    write_csv(out/'route_events.csv', all_routes, common +
              ['event','destination','next_hop_ipv6'])
    write_csv(out/'diagnostics.csv', extracted['diagnostics'], common)
    write_csv(out/'address_map.csv', [{'ipv6': a, 'node': n,
              'mapping_basis': 'startup' if a in all_addresses else 'inferred_unique_64bit_IID'}
              for a,n in extracted['addresses'].items()], ['ipv6','node','mapping_basis'])
    control_fields = common + ['protocol','event','source_node','destination_node',
                              'source_ipv6','destination_ipv6','mode','rank','sequence','prefix','lifetime']
    udp_fields = common + ['event','source_node','source_ipv6','destination_node','sequence',
                          'reported_delay_ms','parent_chain_at_tx','chain_status']
    write_csv(out/'control_timeline.csv', extracted['controls'], control_fields)
    write_csv(out/'udp_events.csv', extracted['udp'], udp_fields)
    write_csv(out/'control_markers.csv', extracted['markers'], common)
    write_csv(out/'control_counts.csv', extracted['control_counts'], ['protocol','event','count'])
    write_csv(out/'dio_rx_candidates.csv', extracted['dio_matches'],
              ['rx_record','rx_time_s','source_node','receiver_node','candidate_count','status',
               'candidate_tx_records','candidate_tx_times_s'])
    write_csv(out/'udp_matches.csv', extracted['udp_matches'],
              ['rx_record','source_node','receiver_node','sequence','rx_time_s','candidate_count',
               'tx_record','tx_time_s','log_time_difference_ms','reported_delay_ms',
               'direction','status','rx_records_for_same_tx'])
    write_csv(out/'udp_tx_without_unique_rx.csv', extracted['udp_unmatched'], udp_fields + ['status'])
    combined = []
    for group, name in [(extracted['controls'],'control'),(all_parents,'parent'),
                        (all_routes,'route'),(extracted['udp'],'udp'),
                        (extracted['diagnostics'],'diagnostic')]:
        combined.extend({**e, 'category': name} for e in group)
    combined.sort(key=lambda e:e['record'])
    write_csv(out/'timeline_all.csv', combined,
              common + ['category','protocol','event','source_node','destination_node'])
    summary['source_file'] = Path(source).name
    summary['timeline_scope'] = 'seluruh_log; snapshot_hanya_untuk_parent_snapshot'
    summary['dio_match_window_s'] = DIO_MATCH_WINDOW_S
    summary['udp_match_window_s'] = UDP_MATCH_WINDOW_S
    summary['full_log_parent_events'] = len(all_parents)
    summary['full_log_udp_events'] = dict(Counter(e['event'] for e in extracted['udp']))
    summary['full_log_route_capacity_errors'] = full_summary['route_capacity_errors']
    (out/'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    (out/'README.txt').write_text(
        'parent_snapshot.csv = rekonstruksi parent terakhir sampai snapshot, bukan '
        'dump routing table IPv6 dan bukan jaminan konektivitas/pengiriman paket.\n'
        'route_events.csv = riwayat pesan rute DAO, BUKAN tabel downward aktif. '
        'Lifetime dan penghapusan/expiry yang tidak tercatat tidak dapat direkonstruksi.\n'
        'record = nomor rekaman data CSV (header tidak dihitung).\n'
        'Link-local berasal dari startup. Global IPv6 dipetakan dengan IID 64-bit '
        'yang sama dan unik; ditandai inferred_unique_64bit_IID. Ini mengasumsikan '
        'global dan link-local node memakai IID sama. Bukan digit akhir IP.\n'
        'MM:SS rollover diasumsikan satu jam sesuai WRAP_SECONDS. '
        'Tidak mendukung log gabungan run, restart, atau pergantian root.\n'
        'Tidak memakai candidate parent atau pesan HO sebagai bukti parent terpilih.\n'
        'Semua timeline mencakup SELURUH log; snapshot hanya mengatur parent_snapshot.\n'
        'control_timeline: TX/RX adalah pesan inti yang eksplisit. OTHER tidak dihitung '
        'sebagai TX/RX. DAO-ACK yang formatnya belum dikenali disimpan utuh.\n'
        'control_markers: MMDEW-CTRL terpisah agar tidak dihitung dua kali.\n'
        'dio_rx_candidates: kandidat berbasis pengirim, penerima, dan jendela waktu. '
        'Bahkan kandidat tunggal BUKAN pasangan paket DIO terkonfirmasi.\n'
        'udp_matches: cocok source + sequence + waktu. Selisih waktu LOG bukan '
        'pengganti reported_delay_ms dari aplikasi. RX berulang diberi hitungan.\n'
        'udp_tx_without_unique_rx bukan jumlah packet loss; logging bisa tidak lengkap.\n'
        'parent_chain_at_tx adalah rekonstruksi parent saat TX, bukan trace per-hop.\n'
        'UDP_NO_CONNECTION tidak dihitung sebagai penerimaan aplikasi. '
        'Tidak menghitung PDR dari log yang kelengkapannya belum diverifikasi.\n'
        'timeline_all dapat memuat rekaman sama pada beberapa kategori. '
        'Jangan gunakan jumlah barisnya sebagai jumlah paket.\n',
        encoding='utf-8')
    archive = out.with_suffix('.zip')
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.iterdir()):
            z.write(p, p.name)
    print(json.dumps(summary, indent=2))
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(table))
        display(pd.DataFrame(extracted['control_counts']))
        display(pd.DataFrame(extracted['controls']).head(20))
        display(pd.DataFrame(extracted['udp_matches']).head(20))
    except ImportError:
        for row in table:
            print(row)
    print('Output:', archive.resolve())
    if INPUT_PATH is None:
        files.download(str(archive))
    return summary


if __name__ == '__main__':
    main()


{
  "root": 30,
  "snapshot_s": 3600.0,
  "log_end_s": 3600.069,
  "nodes_observed": 30,
  "parent_events": 139,
  "chains_to_root": 29,
  "route_capacity_errors": 104,
  "source_file": "DTWRL301.csv",
  "timeline_scope": "seluruh_log; snapshot_hanya_untuk_parent_snapshot",
  "dio_match_window_s": 1.0,
  "udp_match_window_s": 120.0,
  "full_log_parent_events": 139,
  "full_log_udp_events": {
    "TX": 3390,
    "RX": 475,
    "UDP_NO_CONNECTION": 1289,
    "SUMMARY": 60
  },
  "full_log_route_capacity_errors": 104
}


,snapshot_s,node,preferred_parent,parent_ipv6,path_to_root,hops,status,last_parent_event_s,evidence_record
0,3600.0,1,30,fe80::c30c:0:0:1e,1 -> 30,1,rantai_menuju_root,4.573,415
1,3600.0,2,29,fe80::c30c:0:0:1d,2 -> 29 -> 30,2,rantai_menuju_root,3155.437,236930
2,3600.0,3,30,fe80::c30c:0:0:1e,3 -> 30,1,rantai_menuju_root,4.572,414
3,3600.0,4,29,fe80::c30c:0:0:1d,4 -> 29 -> 30,2,rantai_menuju_root,490.939,52294
4,3600.0,5,29,fe80::c30c:0:0:1d,5 -> 29 -> 30,2,rantai_menuju_root,14.077,1444
5,3600.0,6,3,fe80::c30c:0:0:3,6 -> 3 -> 30,2,rantai_menuju_root,40.632,6525
6,3600.0,7,12,fe80::c30c:0:0:c,7 -> 12 -> 30,2,rantai_menuju_root,1595.299,135161
7,3600.0,8,12,fe80::c30c:0:0:c,8 -> 12 -> 30,2,rantai_menuju_root,1591.592,134843
8,3600.0,9,12,fe80::c30c:0:0:c,9 -> 12 -> 30,2,rantai_menuju_root,1591.6,134845
9,3600.0,10,22,fe80::c30c:0:0:16,10 -> 22 -> 30,2,rantai_menuju_root,887.919,84339


,protocol,event,count
0,DAO,NO_PATH_NOTICE,195
1,DAO,OTHER,1670
2,DAO,RX,713
3,DAO,TX,293
4,DAO,TX_NO_PATH,165
5,DIO,OTHER,2701
6,DIO,RX,8663
7,DIO,TX,3849
8,DIS,RX,181
9,DIS,TX,34


,record,time_raw,time_s,node,message,protocol,event,source_node,destination_node,source_ipv6,destination_ipv6,mode,rank,sequence,prefix,lifetime
0,85,00:01.603,1.603,30,[INFO: RPL ] Scheduling DIO timer 372 ti...,DIO,OTHER,,,,,,,,,
1,365,00:02.524,2.524,11,[INFO: RPL ] Sending a DIS to ff02::1a,DIS,TX,11,,,ff02::1a,multicast,,,,
2,367,00:02.716,2.716,1,[INFO: RPL ] Sending a DIS to ff02::1a,DIS,TX,1,,,ff02::1a,multicast,,,,
3,369,00:02.735,2.735,7,[INFO: RPL ] Sending a DIS to ff02::1a,DIS,TX,7,,,ff02::1a,multicast,,,,
4,371,00:02.752,2.752,11,[INFO: RPL ] Received a DIS from fe80::c...,DIS,RX,1,11,fe80::c30c:0:0:1,,,,,,
5,372,00:02.772,2.772,11,[INFO: RPL ] Received a DIS from fe80::c...,DIS,RX,7,11,fe80::c30c:0:0:7,,,,,,
6,373,00:03.042,3.042,9,[INFO: RPL ] Sending a DIS to ff02::1a,DIS,TX,9,,,ff02::1a,multicast,,,,
7,375,00:03.049,3.049,5,[INFO: RPL ] Sending a DIS to ff02::1a,DIS,TX,5,,,ff02::1a,multicast,,,,
8,377,00:03.085,3.085,1,[INFO: RPL ] Received a DIS from fe80::c...,DIS,RX,5,1,fe80::c30c:0:0:5,,,,,,
9,378,00:03.233,3.233,3,[INFO: RPL ] Sending a DIS to ff02::1a,DIS,TX,3,,,ff02::1a,multicast,,,,


,rx_record,source_node,receiver_node,sequence,rx_time_s,candidate_count,tx_record,tx_time_s,log_time_difference_ms,reported_delay_ms,direction,status,rx_records_for_same_tx
0,2804,11,30,0,32.385,1,2799,32.343,42.0,,upward_ke_root,cocok_source_sequence_waktu,1
1,2841,1,30,0,32.555,1,2837,32.535,20.0,156.0,upward_ke_root,cocok_source_sequence_waktu,1
2,2843,27,30,0,32.567,1,2838,32.538,29.0,164.0,upward_ke_root,cocok_source_sequence_waktu,1
3,2860,15,30,0,32.675,1,2851,32.615,60.0,273.0,upward_ke_root,cocok_source_sequence_waktu,1
4,2997,17,30,0,33.005,1,2984,32.962,43.0,601.0,upward_ke_root,cocok_source_sequence_waktu,1
5,2999,9,30,0,33.017,1,2949,32.861,156.0,617.0,upward_ke_root,cocok_source_sequence_waktu,1
6,3080,21,30,0,33.349,1,3011,33.059,290.0,945.0,upward_ke_root,cocok_source_sequence_waktu,1
7,3085,3,30,0,33.424,1,3007,33.052,372.0,1023.0,upward_ke_root,cocok_source_sequence_waktu,1
8,11516,28,30,0,62.723,1,11465,62.608,115.0,7.0,upward_ke_root,cocok_source_sequence_waktu,1
9,11550,15,30,1,62.911,1,11536,62.862,49.0,265.0,upward_ke_root,cocok_source_sequence_waktu,1


Output: /content/hasil_routing/DTWRL301_csv_20261027_001155_389044.zip


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
